In [3]:
import copy
import json
import math
import os
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from transformers import ASTForAudioClassification


# =========================================================
# 1. Mount Drive and copy dataset local
# =========================================================
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/kaggle_cs3780_sp26")
DRIVE_DATA_DIR = DRIVE_ROOT / "lowdim_export" / "reduced_64x32"
LOCAL_ROOT = Path("/content/local_data")
LOCAL_DATA_DIR = LOCAL_ROOT / "reduced_64x32"

if LOCAL_DATA_DIR.exists():
    print(f"Local dataset already exists at {LOCAL_DATA_DIR}")
else:
    LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Copying dataset from {DRIVE_DATA_DIR} to {LOCAL_DATA_DIR} ...")
    shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
    print("Copy complete.")

print("Local train dir:", LOCAL_DATA_DIR / "train")
print("Local test dir:", LOCAL_DATA_DIR / "test")
print("Local solution:", LOCAL_DATA_DIR / "solution.csv")


# =========================================================
# 2. Config
# =========================================================
BASE_DATA_ROOT = LOCAL_ROOT
TRAIN_SUBDIR = Path("reduced_64x32/train")
TEST_SUBDIR = Path("reduced_64x32/test")
SOLUTION_CSV = LOCAL_DATA_DIR / "solution.csv"

OUTPUT_DIR = DRIVE_ROOT / "ast_best_png_cv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"

# AST-style spectrogram size
# final tensor shape = (time_bins, mel_bins)
TIME_BINS = 1024
MEL_BINS = 128

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = torch.cuda.is_available()

N_SPLITS = 5
SEED = 2026

BATCH_SIZE = 12
EPOCHS = 14
PATIENCE = 4
FREEZE_BACKBONE_EPOCHS = 1

LR_HEAD = 1e-4
LR_BACKBONE = 1e-5
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
GRAD_CLIP_NORM = 1.0
USE_AMP = True

EMA_DECAY = 0.999

# polarity ensemble:
# run once with False, once with True if you want even stronger ensemble
INVERT_SPEC = False

# enable TTA on validation/test predictions
USE_TTA = True

print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# =========================================================
# 3. Utils
# =========================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


def save_json(obj, path: Path):
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k, v in self.ema.state_dict().items():
            if v.dtype.is_floating_point:
                v.copy_(v * self.decay + msd[k].detach() * (1.0 - self.decay))
            else:
                v.copy_(msd[k])


def cosine_lr_lambda(current_epoch, total_epochs, warmup_epochs=1):
    if current_epoch < warmup_epochs:
        return float(current_epoch + 1) / float(max(1, warmup_epochs))
    progress = (current_epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
    return 0.5 * (1.0 + math.cos(math.pi * progress))


def build_model(num_classes: int):
    model = ASTForAudioClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_classes,
        ignore_mismatched_sizes=True,
    )
    return model


def freeze_backbone_except_head(model):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True


def unfreeze_all(model):
    for p in model.parameters():
        p.requires_grad = True


def make_optimizer(model, lr):
    return torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr,
        weight_decay=WEIGHT_DECAY,
    )


# =========================================================
# 4. PNG spectrogram -> AST tensor
# =========================================================
def png_to_ast_tensor(
    img: Image.Image,
    train: bool = False,
    invert_spec: bool = False,
    time_bins: int = TIME_BINS,
    mel_bins: int = MEL_BINS,
):
    # grayscale
    img = img.convert("L")

    # resize to AST target size: image shape (mel_bins, time_bins)
    img = img.resize((time_bins, mel_bins), resample=Image.BILINEAR)
    arr = np.asarray(img).astype(np.float32) / 255.0

    if invert_spec:
        arr = 1.0 - arr

    if train:
        # small time shift
        if random.random() < 0.7:
            shift = random.randint(-16, 16)
            arr = np.roll(arr, shift=shift, axis=1)

        # small freq shift
        if random.random() < 0.3:
            shift = random.randint(-4, 4)
            arr = np.roll(arr, shift=shift, axis=0)

        # time mask
        if random.random() < 0.6:
            width = random.randint(12, 80)
            start = random.randint(0, max(0, time_bins - width))
            arr[:, start:start + width] = 0.0

        # freq mask
        if random.random() < 0.6:
            height = random.randint(6, 24)
            start = random.randint(0, max(0, mel_bins - height))
            arr[start:start + height, :] = 0.0

        # mild contrast jitter
        if random.random() < 0.5:
            scale = random.uniform(0.85, 1.20)
            bias = random.uniform(-0.05, 0.05)
            arr = np.clip(arr * scale + bias, 0.0, 1.0)

    # per-image standardization
    mean = arr.mean()
    std = arr.std()
    arr = (arr - mean) / (std + 1e-6)

    # AST expects (time, mel), not (mel, time)
    arr = arr.T
    return torch.tensor(arr, dtype=torch.float32)


# =========================================================
# 5. Datasets
# =========================================================
class FullTrainDataset:
    def __init__(self, root_dir: Path):
        self.root_dir = root_dir
        self.classes = sorted([p.name for p in root_dir.iterdir() if p.is_dir()])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.idx_to_class = {i: c for c, i in self.class_to_idx.items()}

        self.samples = []
        for cls_name in self.classes:
            class_dir = root_dir / cls_name
            for img_path in sorted(class_dir.rglob("*.png")):
                self.samples.append((img_path, self.class_to_idx[cls_name]))

        if len(self.samples) == 0:
            raise RuntimeError(f"No PNG files found under {root_dir}")

        self.labels = [y for _, y in self.samples]


class TrainValFoldDataset(Dataset):
    def __init__(self, samples, train: bool, invert_spec: bool):
        self.samples = samples
        self.train = train
        self.invert_spec = invert_spec

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            x = png_to_ast_tensor(img, train=self.train, invert_spec=self.invert_spec)
        return x, label


class TestDataset(Dataset):
    def __init__(self, test_dir: Path, invert_spec: bool):
        self.image_paths = sorted(test_dir.rglob("*.png"))
        self.invert_spec = invert_spec
        if len(self.image_paths) == 0:
            raise RuntimeError(f"No PNG files found under {test_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        with Image.open(img_path) as img:
            x = png_to_ast_tensor(img, train=False, invert_spec=self.invert_spec)
        return x, img_path.name


# =========================================================
# 6. Eval helpers
# =========================================================
@torch.no_grad()
def predict_logits(model, loader, use_tta=False):
    model.eval()
    all_logits = []
    all_names = []
    use_amp = USE_AMP and DEVICE.type == "cuda"

    for batch in tqdm(loader, leave=False):
        if len(batch) == 2 and isinstance(batch[1][0], str):
            x, names = batch
        else:
            raise RuntimeError("predict_logits expects unlabeled loader")

        x = x.to(DEVICE, non_blocking=True)

        if use_tta:
            variants = [
                x,
                torch.roll(x, shifts=8, dims=1),
                torch.roll(x, shifts=-8, dims=1),
                torch.roll(x, shifts=2, dims=2),
                torch.roll(x, shifts=-2, dims=2),
            ]
            logits_sum = 0
            for v in variants:
                with torch.autocast(device_type="cuda", enabled=use_amp):
                    out = model(input_values=v)
                    logits_sum += out.logits
            logits = logits_sum / len(variants)
        else:
            with torch.autocast(device_type="cuda", enabled=use_amp):
                out = model(input_values=x)
                logits = out.logits

        all_logits.append(logits.float().cpu())
        all_names.extend(list(names))

    return torch.cat(all_logits, dim=0).numpy(), all_names


@torch.no_grad()
def eval_epoch(model, loader, criterion, use_tta=False):
    model.eval()
    all_logits = []
    all_labels = []
    total_loss = 0.0
    use_amp = USE_AMP and DEVICE.type == "cuda"

    for x, y in tqdm(loader, leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        if use_tta:
            variants = [
                x,
                torch.roll(x, shifts=8, dims=1),
                torch.roll(x, shifts=-8, dims=1),
                torch.roll(x, shifts=2, dims=2),
                torch.roll(x, shifts=-2, dims=2),
            ]
            logits_sum = 0
            for v in variants:
                with torch.autocast(device_type="cuda", enabled=use_amp):
                    out = model(input_values=v)
                    logits_sum += out.logits
            logits = logits_sum / len(variants)
        else:
            with torch.autocast(device_type="cuda", enabled=use_amp):
                out = model(input_values=x)
                logits = out.logits

        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)

        all_logits.append(logits.float().cpu())
        all_labels.append(y.cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    preds = logits.argmax(axis=1)
    acc = accuracy_score(labels, preds)
    loss = total_loss / len(loader.dataset)
    return loss, acc, logits, labels


def train_epoch(model, ema, loader, optimizer, scaler, criterion):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    use_amp = USE_AMP and DEVICE.type == "cuda"

    for x, y in tqdm(loader, leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", enabled=use_amp):
            out = model(input_values=x)
            logits = out.logits
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        ema.update(model)

        total_loss += loss.item() * x.size(0)
        all_preds.extend(logits.argmax(dim=1).detach().cpu().numpy())
        all_labels.extend(y.detach().cpu().numpy())

    return total_loss / len(loader.dataset), accuracy_score(all_labels, all_preds)


# =========================================================
# 7. Data
# =========================================================
train_dir = BASE_DATA_ROOT / TRAIN_SUBDIR
test_dir = BASE_DATA_ROOT / TEST_SUBDIR
solution_df = pd.read_csv(SOLUTION_CSV)
solution_map = dict(zip(solution_df["file_name"], solution_df["label"]))

full_train = FullTrainDataset(train_dir)
num_classes = len(full_train.classes)

print("num_classes =", num_classes)
print("num_train =", len(full_train.samples))


# =========================================================
# 8. K-fold training
# =========================================================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_logits = np.zeros((len(full_train.samples), num_classes), dtype=np.float32)
fold_summaries = []

test_dataset = TestDataset(test_dir, invert_spec=INVERT_SPEC)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=PIN_MEMORY,
    persistent_workers=True,
)
test_logits_accum = None
test_file_order = None

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.arange(len(full_train.samples)), full_train.labels), start=1):
    print("\n" + "=" * 100)
    print(f"Fold {fold}/{N_SPLITS}")

    fold_dir = OUTPUT_DIR / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    train_samples = [full_train.samples[i] for i in tr_idx]
    val_samples = [full_train.samples[i] for i in va_idx]

    train_ds = TrainValFoldDataset(train_samples, train=True, invert_spec=INVERT_SPEC)
    val_ds = TrainValFoldDataset(val_samples, train=False, invert_spec=INVERT_SPEC)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=PIN_MEMORY,
        persistent_workers=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=PIN_MEMORY,
        persistent_workers=True,
    )

    model = build_model(num_classes).to(DEVICE)
    ema = ModelEMA(model, decay=EMA_DECAY)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    freeze_backbone_except_head(model)
    optimizer = make_optimizer(model, LR_HEAD)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda ep: cosine_lr_lambda(ep, EPOCHS, warmup_epochs=1)
    )
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE.type == "cuda"))

    best_val_acc = -1.0
    best_epoch = -1
    best_state = None
    best_val_logits = None
    bad_epochs = 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        if epoch == FREEZE_BACKBONE_EPOCHS + 1:
            print(f"[fold {fold}] unfreezing backbone")
            unfreeze_all(model)
            optimizer = make_optimizer(model, LR_BACKBONE)
            scheduler = torch.optim.lr_scheduler.LambdaLR(
                optimizer, lr_lambda=lambda ep: cosine_lr_lambda(ep, max(EPOCHS - epoch + 1, 1), warmup_epochs=1)
            )

        print(f"\n[fold {fold}] epoch {epoch}/{EPOCHS}")

        train_loss, train_acc = train_epoch(model, ema, train_loader, optimizer, scaler, criterion)
        val_loss, val_acc, val_logits, val_labels = eval_epoch(ema.ema, val_loader, criterion, use_tta=USE_TTA)

        history.append({
            "epoch": epoch,
            "train_loss": float(train_loss),
            "train_acc": float(train_acc),
            "val_loss": float(val_loss),
            "val_acc": float(val_acc),
            "lr": float(optimizer.param_groups[0]["lr"]),
        })

        print(
            f"[fold {fold}] "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} "
            f"lr={optimizer.param_groups[0]['lr']:.2e}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_state = copy.deepcopy(ema.ema.state_dict())
            best_val_logits = val_logits.copy()
            bad_epochs = 0
        else:
            bad_epochs += 1

        scheduler.step()

        if bad_epochs >= PATIENCE:
            print(f"[fold {fold}] early stopping")
            break

    model.load_state_dict(best_state)
    ema.ema.load_state_dict(best_state)

    oof_logits[va_idx] = best_val_logits

    # test prediction for this fold
    fold_test_logits, fold_names = predict_logits(ema.ema, test_loader, use_tta=USE_TTA)
    np.save(fold_dir / "test_logits.npy", fold_test_logits)

    if test_logits_accum is None:
        test_logits_accum = fold_test_logits.astype(np.float64)
        test_file_order = fold_names
    else:
        if test_file_order != fold_names:
            raise RuntimeError("Test file order mismatch across folds.")
        test_logits_accum += fold_test_logits.astype(np.float64)

    fold_summary = {
        "fold": fold,
        "best_val_acc": float(best_val_acc),
        "best_epoch": int(best_epoch),
        "history": history,
    }
    fold_summaries.append(fold_summary)
    save_json(fold_summary, fold_dir / "metrics.json")

# OOF accuracy
oof_preds = oof_logits.argmax(axis=1)
oof_acc = accuracy_score(full_train.labels, oof_preds)
print("\nOOF accuracy:", oof_acc)

# =========================================================
# 9. Test prediction + local eval
# =========================================================
test_logits = test_logits_accum / N_SPLITS
test_preds = test_logits.argmax(axis=1)
pred_labels = [full_train.idx_to_class[int(i)] for i in test_preds]

submission_df = pd.DataFrame({
    "file_name": test_file_order,
    "label": pred_labels,
})
submission_path = OUTPUT_DIR / "submission.csv"
submission_df.to_csv(submission_path, index=False)

# local test accuracy using solution.csv if available
y_true_test = [solution_map[f] for f in test_file_order]
test_acc = float(np.mean(np.array(y_true_test) == np.array(pred_labels)))
print("Local test accuracy:", test_acc)

summary = {
    "model_name": MODEL_NAME,
    "invert_spec": INVERT_SPEC,
    "n_splits": N_SPLITS,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "oof_accuracy": float(oof_acc),
    "local_test_accuracy": float(test_acc),
    "submission_csv": str(submission_path),
    "fold_summaries": fold_summaries,
}
save_json(summary, OUTPUT_DIR / "summary.json")

print("Saved submission to:", submission_path)
print(submission_df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Local dataset already exists at /content/local_data/reduced_64x32
Local train dir: /content/local_data/reduced_64x32/train
Local test dir: /content/local_data/reduced_64x32/test
Local solution: /content/local_data/reduced_64x32/solution.csv
Using device: cuda
GPU: NVIDIA A100-SXM4-80GB
num_classes = 9
num_train = 21898

Fold 1/5


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                       
------------------------+----------+---------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([9, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([9])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.



[fold 1] epoch 1/14


  0%|          | 0/1460 [00:00<?, ?it/s]

...

  0%|          | 0/1460 [00:00<?, ?it/s]

  0%|          | 0/365 [00:00<?, ?it/s]

[fold 5] train_loss=1.1173 train_acc=0.6599 val_loss=1.5359 val_acc=0.5108 lr=0.00e+00


  0%|          | 0/455 [00:00<?, ?it/s]


OOF accuracy: 0.5393186592382866
Local test accuracy: 0.5621562156215621
Saved submission to: /content/drive/MyDrive/kaggle_cs3780_sp26/ast_best_png_cv/submission.csv
  file_name           label
0     1.png  Nocturnal bird
1    10.png          Parrot
2   100.png      Flycatcher
3  1000.png  Nocturnal bird
4  1001.png      Flycatcher
